In [1]:
import pandas as pd

In [48]:
# Load draft data and player data

draft_data = pd.read_csv('2024_draft_leagues_1283_recap_combined.csv')
draft_data

,id,league_id,user_team_id,round,pick,player_id,position,autopicked,time
0,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18
1,1705894,1283,719,1,2,138,DEF,0,8/03/2024 13:18
2,1705901,1283,718,1,3,706,RUC,0,8/03/2024 13:18
3,1705907,1283,717,1,4,459,MID,0,8/03/2024 13:18
4,1705908,1283,713,1,5,443,RUC,0,8/03/2024 13:19
...,...,...,...,...,...,...,...,...,...
179,1706317,1283,717,23,180,503,MID,0,8/03/2024 13:44
180,1706330,1283,713,23,181,263,DEF,0,8/03/2024 13:44
181,1706337,1283,716,23,182,386,MID,0,8/03/2024 13:44
182,1706340,1283,715,23,183,721,FWD,0,8/03/2024 13:45


In [49]:
# Get each player's data

player_data = pd.read_json('../../inputs/player_stats_pack/completestatspack_1.json')
player_data

,playerStats,playerMatchStats
0,"{'player_id': 1, 'team_id': 1, 'opponent_id': ...",NaN
1,"{'player_id': 1, 'team_id': 1, 'opponent_id': ...",NaN
2,"{'player_id': 1, 'team_id': 1, 'opponent_id': ...",NaN
3,"{'player_id': 1, 'team_id': 1, 'opponent_id': ...",NaN
4,"{'player_id': 1, 'team_id': 1, 'opponent_id': ...",NaN
...,...,...
65,"{'player_id': 1, 'team_id': 1, 'opponent_id': ...",NaN
66,"{'player_id': 1, 'team_id': 1, 'opponent_id': ...",NaN
67,"{'player_id': 1, 'team_id': 1, 'opponent_id': ...",NaN
68,"{'player_id': 1, 'team_id': 1, 'opponent_id': ...",NaN


In [50]:
player_data = pd.read_csv('2024_player_stats_current.csv', index_col=0)
# player_data[player_data['last_name'] == "Bontempelli"]
player_data

,player_id,first_name,last_name,team_abbrev,pos_1,pos_2,round,points,played,avg,avg3,avg5,price
feed_id,,,,,,,,,,,,,
1012807,1,Sam,Berry,ADE,MID,NaN,1,80,1,80.0000,80.0000,80.00,226900
1012807,1,Sam,Berry,ADE,MID,NaN,2,52,2,66.0000,66.0000,66.00,226900
1012807,1,Sam,Berry,ADE,MID,NaN,4,33,3,55.0000,55.0000,55.00,243100
1012807,1,Sam,Berry,ADE,MID,NaN,5,56,4,55.2500,47.0000,55.25,244400
1012807,1,Sam,Berry,ADE,MID,NaN,6,31,5,50.4000,40.0000,50.40,235500
...,...,...,...,...,...,...,...,...,...,...,...,...,...
996731,99,Charlie,Curnow,CAR,FWD,NaN,18,89,17,85.6471,58.6667,74.80,394800
996731,99,Charlie,Curnow,CAR,FWD,NaN,19,106,18,86.7778,76.0000,80.00,393700
996731,99,Charlie,Curnow,CAR,FWD,NaN,20,119,19,88.4737,104.6670,80.20,431500


In [51]:
# Calculate median scores grouped by player ID

median_data = (player_data.groupby('player_id')['avg']
               .median()
               .reset_index()
               .rename(columns={'avg': 'median_score'}))
median_data

,player_id,median_score
0,1,57.64775
1,2,43.40000
2,3,68.66665
3,4,48.80000
4,5,57.00000
...,...,...
653,799,68.69050
654,800,65.33330
655,804,39.20000
656,806,39.00000


In [52]:
# Fill NA values with 0
median_data['median_score'] = median_data['median_score'].fillna(0)
median_data

,player_id,median_score
0,1,57.64775
1,2,43.40000
2,3,68.66665
3,4,48.80000
4,5,57.00000
...,...,...
653,799,68.69050
654,800,65.33330
655,804,39.20000
656,806,39.00000


In [53]:
# Filter player data for the latest round and merge with median and draft data
latest_round = player_data['round'].max()
heat_map_data = player_data[player_data['round'] == latest_round]
heat_map_data = heat_map_data.merge(median_data, on='player_id', how='left')
heat_map_data = heat_map_data.merge(draft_data, left_on='player_id', right_on='player_id', how='left')
heat_map_data = heat_map_data.sort_values(by='pick', ascending=True)
heat_map_data

,player_id,first_name,last_name,team_abbrev,pos_1,pos_2,round_x,points,played,avg,...,price,median_score,id,league_id,user_team_id,round_y,pick,position,autopicked,time
348,695,Marcus,Bontempelli,WBD,MID,NaN,24,116,23,126.3910,...,665800,125.4440,1705893.0,1283.0,714.0,1.0,1.0,MID,0.0,8/03/2024 13:18
24,138,Nick,Daicos,COL,DEF,MID,24,186,23,117.1740,...,566700,117.1740,1705894.0,1283.0,719.0,1.0,2.0,DEF,0.0,8/03/2024 13:18
355,706,Tim,English,WBD,RUC,NaN,24,103,22,108.0000,...,521900,111.4725,1705901.0,1283.0,718.0,1.0,3.0,RUC,0.0,8/03/2024 13:18
201,443,Max,Gawn,MEL,RUC,NaN,24,122,21,124.1430,...,637600,128.4000,1705908.0,1283.0,713.0,1.0,5.0,RUC,0.0,8/03/2024 13:19
60,197,Zach,Merrett,ESS,MID,NaN,24,135,23,115.2610,...,584800,118.6000,1705909.0,1283.0,716.0,1.0,6.0,MID,0.0,8/03/2024 13:19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405,84,Darcy,Wilmot,BRL,DEF,NaN,24,87,23,83.3478,...,420500,81.7500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
408,89,Jaxon,Binns,CAR,MID,NaN,24,66,3,45.3333,...,152900,35.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
410,91,Jack,Carroll,CAR,FWD,MID,24,38,15,41.6667,...,206000,48.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
411,94,Alex,Cincotta,CAR,DEF,NaN,24,47,16,48.9375,...,249400,48.6143,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
# -- Prepare Positional Data
positions = ['DEF', 'MID', 'RUC', 'FWD']
pos_count = [8*5, 8*7, 8*1, 8*5]

for i, pos in enumerate(positions):
    # Filter and sort by median score for the given position
    pos_data = heat_map_data[(heat_map_data['pos_1'] == pos) | (heat_map_data['pos_2'] == pos)]
    pos_data = pos_data.sort_values(by='median_score', ascending=False).head(pos_count[i])

    # Calculate mean and standard deviation
    mean_score = pos_data['median_score'].mean()
    stdev_score = pos_data['median_score'].std()

    # Normalize scores
    pos_data[f'{pos}_norm_score'] = ((pos_data['median_score'] - mean_score) / stdev_score).round(3)

    # Add normalized scores to the main dataframe
    heat_map_data = pd.merge(heat_map_data, pos_data[['player_id', f'{pos}_norm_score']], on='player_id', how='left')
heat_map_data

,player_id,first_name,last_name,team_abbrev,pos_1,pos_2,round_x,points,played,avg,...,user_team_id,round_y,pick,position,autopicked,time,DEF_norm_score,MID_norm_score,RUC_norm_score,FWD_norm_score
0,695,Marcus,Bontempelli,WBD,MID,NaN,24,116,23,126.3910,...,714.0,1.0,1.0,MID,0.0,8/03/2024 13:18,NaN,2.126,NaN,NaN
1,138,Nick,Daicos,COL,DEF,MID,24,186,23,117.1740,...,719.0,1.0,2.0,DEF,0.0,8/03/2024 13:18,2.352,1.371,NaN,NaN
2,706,Tim,English,WBD,RUC,NaN,24,103,22,108.0000,...,718.0,1.0,3.0,RUC,0.0,8/03/2024 13:18,NaN,NaN,0.083,NaN
3,443,Max,Gawn,MEL,RUC,NaN,24,122,21,124.1430,...,713.0,1.0,5.0,RUC,0.0,8/03/2024 13:19,NaN,NaN,2.282,NaN
4,197,Zach,Merrett,ESS,MID,NaN,24,135,23,115.2610,...,716.0,1.0,6.0,MID,0.0,8/03/2024 13:19,NaN,1.501,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
409,84,Darcy,Wilmot,BRL,DEF,NaN,24,87,23,83.3478,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
410,89,Jaxon,Binns,CAR,MID,NaN,24,66,3,45.3333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
411,91,Jack,Carroll,CAR,FWD,MID,24,38,15,41.6667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
412,94,Alex,Cincotta,CAR,DEF,NaN,24,47,16,48.9375,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [55]:
# -- Calculate Final Score
heat_map_data['final_score'] = heat_map_data[['DEF_norm_score', 'MID_norm_score', 'RUC_norm_score', 'FWD_norm_score']].max(axis=1)
heat_map_data['final_score'] = heat_map_data['final_score'].round(4)
heat_map_data

,player_id,first_name,last_name,team_abbrev,pos_1,pos_2,round_x,points,played,avg,...,round_y,pick,position,autopicked,time,DEF_norm_score,MID_norm_score,RUC_norm_score,FWD_norm_score,final_score
0,695,Marcus,Bontempelli,WBD,MID,NaN,24,116,23,126.3910,...,1.0,1.0,MID,0.0,8/03/2024 13:18,NaN,2.126,NaN,NaN,2.126
1,138,Nick,Daicos,COL,DEF,MID,24,186,23,117.1740,...,1.0,2.0,DEF,0.0,8/03/2024 13:18,2.352,1.371,NaN,NaN,2.352
2,706,Tim,English,WBD,RUC,NaN,24,103,22,108.0000,...,1.0,3.0,RUC,0.0,8/03/2024 13:18,NaN,NaN,0.083,NaN,0.083
3,443,Max,Gawn,MEL,RUC,NaN,24,122,21,124.1430,...,1.0,5.0,RUC,0.0,8/03/2024 13:19,NaN,NaN,2.282,NaN,2.282
4,197,Zach,Merrett,ESS,MID,NaN,24,135,23,115.2610,...,1.0,6.0,MID,0.0,8/03/2024 13:19,NaN,1.501,NaN,NaN,1.501
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
409,84,Darcy,Wilmot,BRL,DEF,NaN,24,87,23,83.3478,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
410,89,Jaxon,Binns,CAR,MID,NaN,24,66,3,45.3333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
411,91,Jack,Carroll,CAR,FWD,MID,24,38,15,41.6667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
412,94,Alex,Cincotta,CAR,DEF,NaN,24,47,16,48.9375,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [56]:
# -- Save Results
# Save the final dataset to a CSV
heat_map_data.to_csv('2024_draft_heat_map.csv', index=False)